# 🚀 [8주차] 한국어 네이티브 모델(A.X) LoRA 파인튜닝 Colab 템플릿

> **💡 Google Colab 사용 안내**
1. 상단 메뉴의 **런타임 > 런타임 유형 변경**에서 하드웨어 가속기를 **L4 GPU**로 설정하세요. (Pro 유저 권장)
2. Colab의 좌측 메뉴 **파일 탐색기(폴더 모양 아이콘)**를 열어 1,000건 이상 증강된 `chatml_dataset.jsonl` 파일을 직접 업로드해 주세요.
3. 업로드 완료 후, 아래 셀(Cell)들을 1번부터 차례대로 실행(Shift+Enter)하시면 파인튜닝이 시작됩니다!

In [ ]:
# 1. Unsloth 패키지 및 핵심 딥러닝 라이브러리 설치 (인스턴스 초기 1회 필수 실행)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
# 2. 변수 및 환경 설정 세팅
import os
import torch
from datasets import load_dataset
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from trl import SFTTrainer
from transformers import TrainingArguments

# 베이스 모델 (추후 A.X 오픈소스 모델 아이디로 교체, 현 예제는 unsloth 호환 최적화 모델 사용)
MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit" 
MAX_SEQ_LENGTH = 2048
DATASET_PATH = "/content/chatml_dataset.jsonl" # 코랩 루트 경로에 업로드한 데이터셋
OUTPUT_DIR = "/content/lora_output" # 어댑터 저장 위치

In [ ]:
# 3. 파운데이션 모델 뼈대 구축 및 4-bit 양자화 로드 (L4 GPU 최적화)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.bfloat16, # L4 GPU는 bfloat16을 네이티브 지원하여 속도가 훨씬 빠릅니다
    load_in_4bit=True,
)

# 4. LoRA 가중치 어댑터(Target Modules) 구성 및 주입
model = FastLanguageModel.get_peft_model(
    model,
    r=16, # Rank 사이즈 지정
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"], # 주요 Attention 모듈에만 타겟팅
    lora_alpha=32,
    lora_dropout=0.05, # 과적합 방지
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

In [ ]:
# 5. 학습 데이터셋 로드 및 ChatML 매핑
if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(f"{DATASET_PATH} 파일이 없습니다! 좌측 파일 창에 업로드했는지 확인하세요.")

dataset = load_dataset("json", data_files=DATASET_PATH, split="train")

# Unsloth의 템플릿 엔진을 통해 ChatML 시스템 매핑을 수행
tokenizer = get_chat_template(
    tokenizer,
    chat_template="chatml",
    mapping={"role": "role", "content": "content", "user": "user", "assistant": "assistant"},
)

def apply_chat_template(examples):
    texts = [tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=False) for msg in examples["messages"]]
    return {"text": texts}

dataset = dataset.map(apply_chat_template, batched=True, num_proc=4) # 프로세스 수 증가

In [ ]:
# 6. SFTTrainer 포맷 설정 및 훈련 개시
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer, # 최신 transformers 버전에 맞춰 tokenizer 대신 processing_class 사용
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=4, # L4 머신 성능에 맞춰 코어 4개 할당
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=8, # L4의 넓은 VRAM(24GB)을 활용해 배치 사이즈를 4배 증가
        gradient_accumulation_steps=1, # 배치 사이즈가 커졌으므로 accumulation 생략하여 연산 속도 극대화
        warmup_steps=10,
        max_steps=100, # 증강된 1000건 데이터에 맞춰 steps 증가
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(), # L4 자동 감지되어 Flash Attention과 함께 적용됨
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
    ),
)

print("🚀 [L4 GPU 최적화] 파인튜닝 훈련 시작...")
trainer_stats = trainer.train()

In [ ]:
# 7. 훈련이 끝난 LoRA 어댑터 가중치 로컬 추출
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"🎉 파인튜닝 로직 완료! 파일이 Colab 내장 디렉토리 {OUTPUT_DIR} 경로에 안전하게 추출되었습니다.")
print("좌측 파일 탭을 열람하여 해당 폴더의 파일들을 다운로드하고, 9주차 GGUF 변환을 대비하세요.")

In [ ]:
# 8. 구글 드라이브 연동 및 백업 (선택사항)
from google.colab import drive
import shutil
import os

# 드라이브 마운트 (실행 시 권한 승인 팝업창이 뜹니다)
drive.mount('/content/drive')

# 사용자 요청에 따라 'Career exploration' 폴더로 백업 경로 설정
DRIVE_DEST_DIR = "/content/drive/MyDrive/Career exploration/lora_output"

if os.path.exists(OUTPUT_DIR):
    # 폴더 통째로 복사 (이미 존재할 경우 덮어쓰기)
    shutil.copytree(OUTPUT_DIR, DRIVE_DEST_DIR, dirs_exist_ok=True)
    print(f"\n✅ 모델 파일이 성공적으로 구글 드라이브에 백업되었습니다!")
    print(f"경로: {DRIVE_DEST_DIR}")
else:
    print(f"\n❌ {OUTPUT_DIR} 폴더를 찾을 수 없습니다. 7번 셀이 정상적으로 실행되었는지 확인하세요.")